# Stage 8 — NVIDIA NeMo AutoModel + LoRA / PEFT

Parameter-efficient adaptation of a vision-language model (VLM) on
HAM10000 using NVIDIA NeMo AutoModel.

> **Research and educational use only.** Not intended for clinical
> diagnosis, medical decision-making, or patient care.

This notebook is the complete record of Stage 8. It documents every
step taken — including hardware checks, tooling research, environment
setup, and model selection — explains why each step was done, and
contains the code needed to reproduce it.

| Step | Content |
|---|---|
| 1 | Hardware check |
| 2 | Investigate NVIDIA tooling and set up the environment |
| 3 | Select a model |
| 4 | NeMo inference baseline (zero-shot, validation data) |
| 5 | LoRA fine-tuning *(to come)* |
| 6 | Evaluate the adapted model *(to come)* |

**Results recorded in markdown** (dates, versions, observed numbers)
are what was observed when the work was done. Re-running on a different
machine or later library versions may produce different values.

## How to Reproduce

1. Create the Stage 8 environment and register it as a Jupyter kernel
   (see Step 2 for why this environment is separate):

   ```bash
   conda env create -f environment-nemo.yml
   conda activate skin-lesion-nemo
   python -m ipykernel install --user --name skin-lesion-nemo --display-name "Python (skin-lesion-nemo)"
   ```

2. Complete Stages 1 and 7 so that `data/raw/ham10000/` and
   `data/processed/multimodal/` exist.
3. Open this notebook from the `notebooks/` directory, select the
   **Python (skin-lesion-nemo)** kernel, and run the cells in order.

`environment-nemo-lock.yml` records the exact package versions used.

### Setup

Imports and project paths used throughout the notebook.

In [ ]:
import subprocess
import sys
from pathlib import Path

import pandas as pd
import torch
import transformers
import nemo_automodel
from PIL import Image

PROJECT_ROOT = Path.cwd().parent

print("Project root:", PROJECT_ROOT)

## Step 1 — Hardware Check

**Why:** GPU memory (VRAM) is the main constraint on VLM fine-tuning. It
determines which models fit, whether LoRA can run in full bf16 precision
or needs quantization (QLoRA), and what batch size and image resolution
are practical. The handoff notes that Stages 1–7 were developed on a
4 GB RTX 3050, so the current machine must be checked rather than
assumed.

Two views are checked:

- `nvidia-smi` reports what the **driver** sees, including the highest
  CUDA version the driver supports.
- PyTorch reports what the **framework** sees, including the CUDA
  version PyTorch was built with. This must be no higher than the
  driver's maximum.

In [ ]:
result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(result.stdout or result.stderr)

In [ ]:
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA (PyTorch build):", torch.version.cuda)

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("GPU:", props.name)
    print("VRAM (GB):", round(props.total_memory / 1024**3, 2))
    print("Compute capability:", f"{props.major}.{props.minor}")
    print("bf16 supported:", torch.cuda.is_bf16_supported())
    print("Architectures in this PyTorch build:", torch.cuda.get_arch_list())

### Step 1 Result (recorded 2026-09-24)

| Component | Value |
|---|---|
| GPU | NVIDIA RTX PRO 5000 Blackwell Generation Laptop GPU |
| VRAM | 24 GB (23.89 GB reported by PyTorch) |
| Compute capability | 12.0 (Blackwell) |
| NVIDIA driver | 596.53 (max CUDA 13.2) |
| PyTorch | 2.14.0+cu130 (CUDA 13.0 build) |
| bf16 support | Yes |
| OS | Windows + WSL2 |

**Interpretation:**

- 24 GB is enough for LoRA on small-to-medium VLMs in bf16, so the
  4 GB workarounds (forced QLoRA, tiny models) are not required.
- Blackwell GPUs (compute capability 12.0) need a PyTorch build for
  CUDA 12.8 or newer. The CUDA 13.0 build satisfies this, and 13.0 is
  below the driver's 13.2 maximum.
- bf16 is the standard mixed-precision format for fine-tuning: half the
  memory of fp32 and more numerically stable than fp16.
- About 3.6 GB of VRAM was already in use by other Python processes
  (open Jupyter kernels) at the time of the check. Close unused kernels
  before training.

## Step 2 — Investigate NVIDIA Tooling

**Why:** NVIDIA's NeMo libraries change quickly, so current
documentation was checked before installing anything, rather than
relying on older tutorials. The goal was to find NVIDIA's recommended
route for LoRA fine-tuning of a Hugging Face VLM on a single GPU, and
to confirm it is compatible with this machine's PyTorch/CUDA stack.

### Findings (researched 2026-09-24)

- **NeMo AutoModel** (`nemo-automodel`, v0.6.0, released 2026-08-26) is
  NVIDIA's current library for fine-tuning Hugging Face LLMs and VLMs.
  It is part of the NeMo Framework. (Megatron-Bridge is NVIDIA's route
  for very large-scale, multi-node training and is not needed here.)
- It loads **Hugging Face checkpoints directly** — the same model IDs
  used in Stage 6.
- Supported training methods include full SFT, **LoRA**, and QLoRA.
- Training runs are defined as **YAML recipes** and launched with
  `automodel <recipe.yaml>`, replacing a hand-written training loop.
- Its model loader, `NeMoAutoModelForImageTextToText`, is documented as
  a drop-in replacement for `transformers.AutoModelForImageTextToText`
  (used in Stage 6) that adds NVIDIA kernel optimizations, LoRA, and
  distributed-training support.

### Compatibility

| Requirement (nemo-automodel 0.6.0) | Main env `skin-lesion-ai` | Compatible? |
|---|---|---|
| Python ≥ 3.10 | 3.11 | Yes |
| `torch>=2.6.0`, CUDA 13.0 wheels | 2.14.0+cu130 | Yes |
| `transformers==5.12.1` (exact pin) | 5.17.0 | **No** — would be downgraded |

**Decision:** create a **separate environment**, `skin-lesion-nemo`,
rather than installing NeMo into `skin-lesion-ai`. This keeps the
environment that produced the Stage 1–7 results unchanged.

Sources:

- [NeMo AutoModel — Vision Language Models](https://docs.nvidia.com/nemo/automodel/model-coverage/vision-language-models/overview)
- [nemo-automodel on PyPI](https://pypi.org/project/nemo-automodel/)
- [NVIDIA-NeMo/Automodel on GitHub](https://github.com/NVIDIA-NeMo/Automodel)

### Environment Setup

The environment was built in this order. **PyTorch is installed first,
pinned to the CUDA 13.0 build**, so that installing NeMo AutoModel does
not pull in a different PyTorch build.

```bash
conda create -n skin-lesion-nemo python=3.11 -y
conda activate skin-lesion-nemo

pip install torch==2.14.0 torchvision --index-url https://download.pytorch.org/whl/cu130

pip install "nemo-automodel[vlm,vlm-media]" ipykernel
pip install num2words==0.5.14
python -m ipykernel install --user --name skin-lesion-nemo --display-name "Python (skin-lesion-nemo)"
```

The result is saved as `environment-nemo.yml` (recipe) and
`environment-nemo-lock.yml` (exact versions), so it can be recreated
with `conda env create -f environment-nemo.yml`.

`num2words` was added during Step 4b: the SmolVLM processor requires it
(to write image-tile positions as words in the prompt), but NeMo
AutoModel does not install it. The version matches the `skin-lesion-ai`
environment.

**Note:** PyTorch's pip wheels include their own CUDA libraries
(cuDNN, cuBLAS, NCCL, and others). No separate CUDA toolkit is needed;
only the NVIDIA driver is installed system-wide.

The next cells verify the environment. PyTorch should still be the
CUDA 13.0 build, and the GPU should run a bf16 computation.

In [ ]:
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__, "| CUDA build:", torch.version.cuda)
print("transformers:", transformers.__version__)
print("NeMo AutoModel:", nemo_automodel.__version__)

import num2words  # required by the SmolVLM processor
print("num2words: installed")

x = torch.randn(1024, 1024, device="cuda", dtype=torch.bfloat16)
print("bf16 matmul on GPU OK:", bool((x @ x).isfinite().all()))

In [ ]:
# Check installed packages for dependency conflicts
result = subprocess.run(
    [sys.executable, "-m", "pip", "check"],
    capture_output=True,
    text=True,
)
print(result.stdout or result.stderr)

### Step 2 Result (recorded 2026-09-24)

| Package | Version |
|---|---|
| PyTorch | 2.14.0+cu130 (unchanged by the NeMo install) |
| transformers | 5.12.1 |
| NeMo AutoModel | 0.6.0 |
| num2words | 0.5.14 (added in Step 4b) |

`pip check` reports `decord 0.6.0 is not supported on this platform`.
`decord` is only used for video decoding; this project uses still
images, so the warning can be ignored.

## Step 3 — Select a Model

**Why:** A model must be (1) supported by NeMo AutoModel, (2) able to
take an image plus instruction and generate text, (3) trainable with
LoRA, and (4) small enough for the available GPU memory.

**Decision:** start with SmolVLM, the model family from Stage 6, so the
fine-tuned result can be compared directly with the Stage 6 zero-shot
baseline. A stronger model (for example Qwen2.5-VL-3B) may be added
later.

The handoff warned not to assume the Stage 6 models are supported by
NeMo. NeMo AutoModel supports models by **architecture** (the Python
model class), not by name, so the next cells check which architecture
each Stage 6 model uses and whether NeMo AutoModel's code refers to it.

In [ ]:
from transformers import AutoConfig

STAGE6_MODELS = [
    "HuggingFaceTB/SmolVLM-256M-Instruct",
    "HuggingFaceTB/SmolVLM2-500M-Video-Instruct",
]

model_architectures = {}

for model_id in STAGE6_MODELS:
    config = AutoConfig.from_pretrained(model_id)
    model_architectures[model_id] = config.architectures[0]
    print(f"{model_id}\n  architecture: {config.architectures[0]} | model_type: {config.model_type}")

In [ ]:
# Search the installed NeMo AutoModel package for references to each architecture
NEMO_PACKAGE_DIR = Path(nemo_automodel.__file__).parent

def files_referencing(class_name):
    return sorted(
        str(path.relative_to(NEMO_PACKAGE_DIR))
        for path in NEMO_PACKAGE_DIR.rglob("*.py")
        if class_name in path.read_text(encoding="utf-8", errors="ignore")
    )

for model_id, architecture in model_architectures.items():
    files = files_referencing(architecture)
    print(f"{architecture} ({model_id})")
    print("  referenced in NeMo AutoModel:", files if files else "NOT FOUND")

### Step 3 Result (recorded 2026-09-24)

| Model | Architecture | Supported by NeMo AutoModel? | Weights on disk |
|---|---|---|---|
| SmolVLM-256M-Instruct | `Idefics3ForConditionalGeneration` | No | 0.5 GB |
| **SmolVLM2-500M-Video-Instruct** | `SmolVLMForConditionalGeneration` | **Yes** | 2.0 GB |

Despite the "SmolVLM" name, the 256M model is built on the older
Idefics3 architecture, which NeMo AutoModel does not support. NVIDIA's
documentation lists `SmolVLMForConditionalGeneration` as supported but
provides no ready-made SmolVLM recipe, so a custom recipe will be
written in Step 5.

**Selected model: `HuggingFaceTB/SmolVLM2-500M-Video-Instruct`**

- Supported architecture.
- Has a Stage 6 zero-shot result to compare against (11.4% strict
  accuracy, 0 invalid outputs, severe collapse onto `akiec` and `bkl`).
  The Stage 8 question becomes: *does LoRA fine-tuning fix the class
  collapse?*
- About 0.5B parameters, roughly 1 GB in bf16, so LoRA training should
  use a small fraction of the 24 GB GPU. Memory is measured in Step 4.
- "Video" in the name means the model can also accept video; still
  images work normally.

## Step 4 — NeMo Inference Baseline

Before any fine-tuning, confirm that SmolVLM2 runs correctly through the
NeMo AutoModel stack and measure its zero-shot performance.

**Why a baseline is needed:**

- It separates pipeline problems (prompt formatting, image handling,
  truncated outputs) from training problems. If later results are poor,
  a working baseline shows the pipeline was not the cause.
- It gives the "before" number that the LoRA result will be compared with.
- It measures GPU memory use, which NVIDIA's documentation does not give.

**Why this is re-run instead of reusing the Stage 6 numbers:**

- **Validation, not test.** Stage 6 used test images. Development
  decisions in Stage 8 use validation data; the test set stays isolated
  until final evaluation.
- **Same prompt as training.** The Stage 7 CSVs use a slightly different
  instruction wording from the Stage 6 prompt. The baseline must use the
  exact instruction the model will be trained on.
- **Same precision as training.** Stage 6 ran in fp16; Stage 8 trains
  in bf16.

This makes the before/after comparison isolate the effect of LoRA.

Step 4 is split into:

- **4a** — prepare the validation sample and conversation format
- **4b** — load SmolVLM2 through NeMo AutoModel and check it runs
- **4c** — run the 35-image baseline and compute metrics

### 4a.1 — Load the validation split

Uses the Stage 7 multimodal validation file, which preserves the
lesion-aware split.

In [ ]:
VAL_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "multimodal"
    / "val.csv"
)

val_df = pd.read_csv(VAL_PATH)

# Image paths are stored relative to the project root
val_df["image_path"] = val_df["image_path"].map(lambda p: str(PROJECT_ROOT / p))

print("Validation records:", len(val_df))
print("Unique instructions:", val_df["instruction"].nunique())
print()
print(val_df["response"].value_counts())

### 4a.2 — Balanced validation sample

Same design as the Stage 6 experiment (5 images per class, seed 42),
drawn from validation instead of test. This fixed sample is used for the
baseline and later for comparing the LoRA-adapted model.

In [ ]:
SAMPLES_PER_CLASS = 5
RANDOM_SEED = 42

val_sample_df = (
    val_df
    .groupby("response", group_keys=False)
    .sample(n=SAMPLES_PER_CLASS, random_state=RANDOM_SEED)
    .reset_index(drop=True)
)

print(val_sample_df["response"].value_counts().sort_index())

### 4a.3 — Convert records to the NeMo AutoModel conversation format

NeMo AutoModel VLM datasets represent each example as a chat
conversation (the same structure as its built-in MedPix dataset):

- **user** turn: the image and the instruction
- **assistant** turn: the target response

During **training** (train split), the assistant turn is included so the
loss can be computed against it and the model's weights are updated.

During **evaluation** (validation split during development, test split
once at the end), the assistant turn is left out; the model generates its
own answer, which is compared with the stored target afterward. The
weights are not changed.

In [ ]:
def to_conversation(row, include_response):
    """Convert one Stage 7 record into the NeMo AutoModel conversation format.

    include_response=True  -> training example (target answer included)
    include_response=False -> evaluation prompt (target answer withheld)
    """
    conversation = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": Image.open(row["image_path"]).convert("RGB")},
                {"type": "text", "text": row["instruction"]},
            ],
        }
    ]

    if include_response:
        conversation.append(
            {
                "role": "assistant",
                "content": [{"type": "text", "text": row["response"]}],
            }
        )

    return conversation

In [ ]:
example = val_sample_df.iloc[0]

train_view = to_conversation(example, include_response=True)
eval_view = to_conversation(example, include_response=False)

print("Image ID:", example["image_id"])
print("Training form roles:  ", [turn["role"] for turn in train_view])
print("Evaluation form roles:", [turn["role"] for turn in eval_view])
print()
print("Instruction:")
print(example["instruction"])
print()
print("Target response (withheld at evaluation):", example["response"])

### 4b.1 — Load SmolVLM2 through NeMo AutoModel

**Why:** this is the first use of NeMo itself. The model is loaded with
`NeMoAutoModelForImageTextToText`, which NVIDIA documents as a drop-in
replacement for the Hugging Face `AutoModelForImageTextToText` loader
used in Stage 6. It returns the same Hugging Face model class, but the
loader also:

- places the model on the GPU automatically
- selects an attention implementation (FlashAttention if installed,
  otherwise PyTorch SDPA)
- tries to apply NVIDIA/Liger kernel optimizations, falling back if they
  are unavailable
- provides the hooks NeMo uses later for LoRA and distributed training

The **processor** (image preprocessing + tokenizer + chat template)
comes from Hugging Face `AutoProcessor`, as in Stage 6.

Warnings expected during loading, all harmless for single-GPU work with
this model:

- `Liger Kernel ... could not import` — optional speed-up kernels are
  not installed; standard kernels are used.
- `Transformer Engine and Apex are not installed` — optional NVIDIA
  libraries for large-scale training.
- `grouped_gemm is not available` — only used by Mixture-of-Experts
  models.
- `pad_token_id must be None or an integer within the vocabulary` —
  a quirk of the model's config file.

In [ ]:
from transformers import AutoProcessor
from nemo_automodel import NeMoAutoModelForImageTextToText

MODEL_ID = "HuggingFaceTB/SmolVLM2-500M-Video-Instruct"

processor = AutoProcessor.from_pretrained(MODEL_ID)

model = NeMoAutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
)
model.eval()

In [ ]:
from collections import Counter

param_dtypes = Counter(str(p.dtype) for p in model.parameters())

print("Model class:", type(model).__name__)
print("Device:", next(model.parameters()).device)
print("Attention implementation:", model.config._attn_implementation)
print("Parameters (millions):", round(sum(p.numel() for p in model.parameters()) / 1e6, 1))
print("Parameter dtypes:", dict(param_dtypes))
print("GPU memory allocated (GB):", round(torch.cuda.memory_allocated() / 1024**3, 2))

### 4b.2 — Why the weights are float32, and the inference precision decision

Although `torch_dtype=torch.bfloat16` was requested, the loaded
parameters are **float32**.

This is deliberate NeMo behavior. The SmolVLM2 checkpoint on Hugging
Face is stored in float32 (2 GB for 0.5B parameters = 4 bytes each).
NeMo AutoModel keeps the **wider** of the checkpoint precision and the
requested precision, so the weights stay float32 as *master weights*.
During training, NeMo computes in bf16 while keeping these float32
master weights — standard **mixed-precision training**, which is more
numerically stable than training directly in bf16.

For **inference**, two options were tested:

1. **bf16 autocast** (float32 weights, bf16 computation). This fails
   with SmolVLM: its image-merging step raises
   `RuntimeError: Index put requires the source and destination dtypes
   match`, because text and image features end up in different
   precisions.
2. **Cast the weights to bf16** with `model.to(torch.bfloat16)`.

The next cells compare float32 and option 2 on one validation image per
class, using the prediction function defined below.

### 4b.3 — Prediction function

Builds the **evaluation form** of the conversation (no assistant turn,
so the answer is withheld), applies the model's chat template with
`add_generation_prompt=True` so the prompt ends with `Assistant:` and
the model writes the answer, then decodes only the newly generated
tokens.

Settings:

- `max_new_tokens=10` — the expected answer is a single abbreviation,
  as in Stage 6.
- `do_sample=False` (greedy decoding) — the model always picks its
  most likely next token, so results are deterministic and
  reproducible.
- The output is normalized the same way as Stage 6: strip whitespace,
  lowercase, remove periods.

In [ ]:
VALID_CLASSES = {"akiec", "bcc", "bkl", "df", "mel", "nv", "vasc"}


def predict(row, model, processor, max_new_tokens=10):
    """Generate the model's answer for one record (answer withheld)."""
    conversation = to_conversation(row, include_response=False)

    inputs = processor.apply_chat_template(
        conversation,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        # transformers 5 expects processor options inside processor_kwargs
        processor_kwargs={"return_tensors": "pt"},
    ).to(model.device)

    # Match the image tensor precision to the model weights
    inputs["pixel_values"] = inputs["pixel_values"].to(model.dtype)

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

    new_tokens = generated_ids[:, inputs["input_ids"].shape[1]:]
    raw_output = processor.batch_decode(new_tokens, skip_special_tokens=True)[0]
    prediction = raw_output.strip().lower().replace(".", "")

    return raw_output, prediction

Inspect the exact prompt the model receives for one record. It should
contain the image placeholder tokens, the Stage 7 instruction, and end
with `Assistant:` — and it must **not** contain the answer.

In [ ]:
check_inputs = processor.apply_chat_template(
    to_conversation(val_sample_df.iloc[0], include_response=False),
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    # transformers 5 expects processor options inside processor_kwargs
    processor_kwargs={"return_tensors": "pt"},
)

prompt_text = processor.tokenizer.decode(check_inputs["input_ids"][0])

print("Prompt length (tokens):", check_inputs["input_ids"].shape[1])
print("Image tensor shape (batch, tiles, channels, height, width):", tuple(check_inputs["pixel_values"].shape))
print()
print("End of prompt:")
print(prompt_text[-420:])

### 4b.4 — Compare float32 and bf16 inference

One validation image per class (7 images). A warm-up call runs first so
that one-time GPU setup does not distort the timing.

In [ ]:
import time

one_per_class_df = val_sample_df.groupby("response").head(1).reset_index(drop=True)


def run_precision_check(label):
    torch.cuda.reset_peak_memory_stats()
    start = time.perf_counter()
    predictions = [predict(row, model, processor)[1] for _, row in one_per_class_df.iterrows()]
    seconds_per_image = (time.perf_counter() - start) / len(one_per_class_df)
    return {
        "precision": label,
        "predictions": predictions,
        "seconds_per_image": round(seconds_per_image, 2),
        "weights_gb": round(torch.cuda.memory_allocated() / 1024**3, 2),
        "peak_gb": round(torch.cuda.max_memory_allocated() / 1024**3, 2),
    }


predict(one_per_class_df.iloc[0], model, processor)  # warm-up

fp32_check = run_precision_check("float32")

model.to(torch.bfloat16)
torch.cuda.empty_cache()

bf16_check = run_precision_check("bf16")

print("True classes:", list(one_per_class_df["response"]))
for check in (fp32_check, bf16_check):
    print(
        f"{check['precision']:8s} predictions: {check['predictions']} | "
        f"{check['seconds_per_image']} s/image | "
        f"weights {check['weights_gb']} GB | peak {check['peak_gb']} GB"
    )
print()
print("Same predictions:", fp32_check["predictions"] == bf16_check["predictions"])
print("Model dtype now:", model.dtype)

### Step 4b Result (recorded 2026-09-24)

| | float32 | bf16 (cast) |
|---|---|---|
| Predictions (one per class) | all `akiec` | all `akiec` |
| Time per image | 0.75–0.80 s | 0.40–0.55 s |
| GPU memory, weights | 1.94 GB | 0.99 GB |
| GPU memory, peak during inference | 2.51 GB | 1.29 GB |

**Findings:**

- **The NeMo pipeline works:** the model loads on the GPU, images reach
  the model (13 image tiles per image), the prompt ends with
  `Assistant:` and contains no answer, and the output is a valid class
  abbreviation.
- **Decision: use bf16 for Stage 8 inference.** It gives identical
  predictions at half the memory and is noticeably faster, and matches the
  bf16 computation used in training.
- **Memory is not a constraint:** peak inference memory is about 1.3 GB
  of the 24 GB available.
- **The Stage 6 class collapse reappears immediately:** all 7 images,
  one from each class, are predicted `akiec`. The one correct answer
  (the `akiec` image) is therefore not evidence of skill. Step 4c
  measures this properly on the full 35-image sample.